# 第13回 映画のジャンルを、年代の地図に（解答例・教員用）


---


## 前回からの接続


第12回は、数の列（気温・降水量）を少ない軸にまとめました。今日のデータは、数ではなく区分（年代とジャンル）です。区分×区分で数えた表を、地図にします。


---


## 今日の分析目標


映画のジャンルの好まれ方は、年代でどう違うか。それを1枚の地図にできるかに答える。

まず、映画1本ごとの一覧から、題名で年代を取り出し、ジャンルを分けて、年代×ジャンルの表を作ります。次に、偏りがなければ何本になるはずか（期待度数）を出し、観測と比べた倍率を見ます。そして、ずれの大きさを標準化残差（standardized residual）で測ります。さらに、ずれの表を特異値分解（singular value decomposition）で縮めて、年代とジャンルを1枚の地図に置きます（対応分析）。最後に、地図で見えたことを、倍率の表で確かめます。

もう1つのねらいは、文字列を区切って分け、1行に詰まった複数の値を、数えられる形に直せるようになることです。お手本ごとに、コードを1行ずつ読み（「1行ずつ読む」）、変数の中身を見ます（「中身を見る」）。


### 公開データの世界

ミネソタ大学のGroupLensが、映画の評価サイトMovieLensのデータを研究用に公開しています。今日は、その小さな版（ml-latest-small）の映画の一覧を使います。再配布は認められていないので、このノートの中で取得元から取ってきます。


### 場面

1930年代の映画と、2010年代の映画では、作られたジャンルが違いそうです。白黒の時代はロマンス、今はアクションでしょうか。印象ではなく、数えた表から読みます。

年代ごとに、どのジャンルが多いか、少ないか。本数の多い年代・ジャンルに引きずられずに、割合の偏りを読み、年代とジャンルの関係を1枚の地図にします。


### 手元のデータ

MovieLensの映画の一覧（Harper & Konstan, 2015）で、9,742本です。1行が1本の映画で、題名に公開年、ジャンルの列には「|」で区切った複数のジャンルが入っています。答えの列はありません。年代とジャンルの組を数えます。


### 最後に出す答え

年代ごとに、どのジャンルが割合として多いか。年代とジャンルの関係は、1枚の地図にするとどう見えるかです。


### 今日の一周（結果だけ先に）

コードの説明はあとにして、今日の解析を、結果だけで一度通して見ておきます。

1. 表を作る：えっ、年代×ジャンルの表の合計は18,544本なのに、映画は9,162本しかありません。
2. 独立なら何本？：年代の本数とジャンルの本数から、偏りがなければ何本かを出し、何倍かを見ます。
3. ずれを測る：多すぎ・少なすぎを標準化残差で測ると、1930年代のロマンス、2010年代のアクションが目立ちます。
4. 地図に縮める：横軸は古い↔新しい。えっ、本数がいちばん多い2000年代は、真ん中にいます。
5. 倍率で確かめる：1930年代はロマンス2.32倍、1980年代はホラー1.51倍、2010年代はアクション1.27倍です。

このあと、この5つの段の中身を、1つずつ開けていきます。


### 今日の問いとやること

| 段 | 問い | やること |
|---|---|---|
| 表を作る（1節） | 一覧からどう表にする？ | 題名から年代を取り出し、ジャンルを分けて、年代×ジャンルの表を作る |
| 独立なら何本？（2節） | 偏りがなければ何本のはず？ | 期待度数を出し、観測と比べた倍率を見る |
| ずれを測る（3節） | どのマスがずれている？ | 標準化残差を色の表で見る |
| 地図に縮める（4節） | 1枚の地図にできるか？ | 残差の表を特異値分解し、年代とジャンルを同じ地図に置く |
| 倍率で確かめる（5節） | 地図は本当か？ | 地図の近さを倍率の表で確かめ、年代ごとに1文にする |


### 今日の進め方

段ごとに、まずスライドで、その段の結論までを通して聞きます。段の終わりにノートへ移り、その段の節のセルを上から実行して、スライドと同じ数が出るかを確かめます。そして、段ごとに1つあるTODOに答えます。

TODOは、実行の前に結果を予想して書き、1か所を替えて実行し、予想と比べる形です。予想が外れても、かまいません。


---


## 学習ゴール


この回を終えると、次のことができるようになります。

- 文字列を区切って分け（`split`）、1行を何行にもして（`explode`）、区分×区分の表を作れる
- 表の合計が、数えたものの数と合うかを確かめられる
- 期待度数（expected frequency）と倍率・標準化残差で、どこが多すぎ・少なすぎかを読める
- 残差（residual）の表を特異値分解し、行と列を1枚の地図に置いて読める
- 地図で見えたことを、倍率の表で確かめて言葉にできる


---


## 0. 準備


### 最初に実行する：道具を呼び出す

- 役割：このノートで使う道具（ライブラリ）を呼び出す
- 入力：なし
- 出力：なし（終わるとセルの左に ✓ が付く）


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


### 図の日本語を表示できるようにする

- 役割：図の中の日本語が □ にならないようにし、図の大きさと字の大きさをそろえる
- 入力：なし
- 出力：「セットアップ完了」の表示

`try:`から先は、日本語表示の道具が動かなくなったときの保険で、中身は読まなくてかまいません。


In [ ]:
# 日本語が図の中で □ になるのを防ぐ
!pip install -q japanize-matplotlib
try:
    import japanize_matplotlib
# japanize が動かなくなったときの保険：同梱フォントを直接登録する
except Exception:
    from importlib.util import find_spec
    from matplotlib import font_manager as fm
    from pathlib import Path
    spec = find_spec('japanize_matplotlib')
    ttf = None
    if spec:
        ttf = next(Path(spec.origin).parent.rglob('*.ttf'), None)
    if ttf:
        fm.fontManager.addfont(str(ttf))
        name = fm.FontProperties(fname=str(ttf)).get_name()
        plt.rcParams['font.family'] = name

plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['font.size'] = 13
plt.rcParams['axes.unicode_minus'] = False  # マイナス記号の化けを防ぐ
print('セットアップ完了')


---


## 1. 一覧から、どう表にする？


ここから、段1「表を作る」（1節）です。問いは、映画1本ごとの一覧から、年代×ジャンルの表をどう作るか、です。


### お手本：取ってきて展開する

- 役割：GroupLensから映画の一覧のzipを取ってきて、展開する
- 入力：作業場所`WORK`と、取得元`URL`
- 出力：フォルダ`ml-latest-small`（5つのファイル）


In [ ]:
import os
import shutil
import urllib.request
WORK = '/content'   # Colab の作業場所
URL = ('https://files.grouplens.org/datasets/movielens/'
       'ml-latest-small.zip')
if not os.path.exists(f'{WORK}/ml-latest-small'):
    urllib.request.urlretrieve(URL, f'{WORK}/ml.zip')
    shutil.unpack_archive(f'{WORK}/ml.zip', WORK)
print(sorted(os.listdir(f'{WORK}/ml-latest-small')))


#### 1行ずつ読む

- `import`の行：フォルダ（`os`）・zip（`shutil`）・URL（`urllib`）を扱う道具
- `WORK`と`URL`の行：作業場所と取得元（Colabでは、作業場所は`/content`になっている）
- `if`の行：まだ展開したフォルダが無いときだけ、取ってきて（`urlretrieve`）、展開する（`unpack_archive`）
- `print`の行：フォルダの中の5つのファイル。使うのは`movies.csv`だけ


### お手本：映画の一覧を読む


In [ ]:
movies = pd.read_csv(f'{WORK}/ml-latest-small/movies.csv')
print(len(movies))
movies.head(3)


#### 1行ずつ読む

- 1行目：一覧を読む
- 2行目：9,742本
- 3行目：先頭3本。1本目のトイ・ストーリーは、5つのジャンルを持つ


### お手本：公開年と年代

公開年は、題名の最後の「(1995)」の中にあります。文字列の決まった形（かっこの中の4けたの数）を探して取り出します。その形を書く文字の並びを、正規表現（regular expression）といいます。今日は1つだけ使います。

- 役割：題名から公開年を取り出し、年代の列を作る
- 入力：表`movies`
- 出力：表`m`（年・年代の列つき）


In [ ]:
m = movies.copy()
m['年'] = m['title'].str.extract(r'\((\d{4})\)\s*$')[0].astype(float)
m = m.dropna(subset=['年'])
m['年代'] = (m['年'] // 10 * 10).astype(int)
m = m[m['年代'] >= 1930]
print(len(m))


#### 1行ずつ読む

- 1行目：写しを作る（`movies`を壊さない）
- 2行目：題名の最後の「(4けたの数)」を取り出し（`str.extract`）、数にする。中身は読み飛ばしてよい
- 3行目：公開年が書かれていない映画を外す
- 4行目：10で割って切り捨て、10を掛けて年代にする（1995 → 1990）
- 5・6行目：1930年代より前は本数が少ないので外す。残りは9,682本


### 文字列を分ける：最小の例

ジャンルの列には「Adventure|Comedy」のように、複数のジャンルが「|」で区切って入っています。このままでは、ジャンルごとに数えられません。「|」で分けて、1本の映画を、ジャンルの数だけの行にします。


In [ ]:
s = 'Adventure|Comedy|Romance'
print(s.split('|'))
print(len(s.split('|')))


#### 1行ずつ読む

- 1行目：「|」で区切った文字列
- 2行目：`split('|')`で分けると、リストになる
- 3行目：3つ


### お手本：ジャンルを分ける

列の1つずつを分けてリストにし（`str.split`）、リストの1つずつを1行にします（`explode`）。

- 役割：ジャンルの列を分け、1本の映画をジャンルの数だけの行にする
- 入力：表`m`
- 出力：表`g`


In [ ]:
g = m.copy()
g['ジャンル'] = g['genres'].str.split('|')
g = g.explode('ジャンル')
print(len(m), len(g))
g[['title', 'ジャンル']].head(5)


#### 1行ずつ読む

- 1行目：写しを作る
- 2行目：ジャンルの列を、1本ずつ「|」で分けてリストにする（`str.split`は列のすべての行に`split`を当てる）
- 3行目：リストの1つずつを1行にする（`explode`）。題名や年代は、同じ値が写される
- 4・5行目：行の数は9,682から21,972に増えた。先頭5行を見る


#### 中身を見る：トイ・ストーリー


In [ ]:
g[g['title'] == 'Toy Story (1995)'][['title', '年代', 'ジャンル']]


1本の映画が、5つのジャンルの5行になりました。年代は5行とも同じです。


> 今日の重要概念（Python）：文字列を分ける
>
> 1つのマスに詰めた複数の値は、区切りの文字で分け（split）、1つずつを1行にする（explode）。そうすると、数えられる形になる。


### お手本：年代×ジャンルの表

- 役割：上位10ジャンルに絞り、年代×ジャンルの組ごとに数える
- 入力：表`g`
- 出力：観測度数（observed frequency）の表`O`（9年代×10ジャンル）


In [ ]:
g_all = g
top = g['ジャンル'].value_counts().index[:10]
g = g[g['ジャンル'].isin(top)]
O = pd.crosstab(g['年代'], g['ジャンル'])
print(O.shape, O.values.sum(), g['movieId'].nunique())


#### 1行ずつ読む

- 1行目：絞る前の表を`g_all`にとっておく（TODO①で使う）
- 2・3行目：本数の多い上位10ジャンルを選び、その行だけ残す（`isin`）
- 4行目：年代×ジャンルの組ごとに数える（`pd.crosstab`、第7回）。観測度数の表`O`
- 5行目：形、表の合計、映画の本数（`nunique`は重なりを除いた数）

えっ、表の合計は18,544なのに、映画は9,162本です。7割の映画が2つ以上のジャンルを持ち、重ねて数えられています。表の1つの数は、1本の別の映画とは限りません。


In [ ]:
O


In [ ]:
print(O.sum(axis=1).tolist())


In [ ]:
O.sum(axis=1).plot.bar()
plt.ylabel('表の数（重ねて数えた本数）')
plt.show()


新しい年代ほど、本数が多くなっています。2000年代がいちばん多い年代です。


### 結果を見る前に、予想を書く

1930年代のロマンスは、全体と比べて多いでしょうか。予想と、その理由を1行で書いてください。


（ここに予想を書く）


### TODO①：ジャンルの数を替える（予想 → 実行 → 比べる）

下のセルは、上位を選ぶ前の表`g_all`から上位のジャンルを選んで、年代×ジャンルの表を作り、形と合計を表示します。

まず、予想を書いてください。6ジャンルにすると、表の合計はどれだけ減るでしょうか。


（ここに予想を書く）


予想を書いたら、セルの`10`を`6`に替えて実行してください。


In [ ]:
# TODO: 10 を 6 に替えて実行する

top_x = g_all['ジャンル'].value_counts().index[:6]
g_x = g_all[g_all['ジャンル'].isin(top_x)]
O_x = pd.crosstab(g_x['年代'], g_x['ジャンル'])
print(O_x.shape, O_x.values.sum())


### やってみよう：予想と比べる

実行した結果を、予想と比べてください。表の合計は、どれだけ減りましたか。


（ここに答えを書く）


6ジャンルで14,631です。少ないジャンルの4つを外しても、2割ほどしか減りません。上位のジャンルに、数が集まっています。


### ここまでで分かったこと

| 段 | 分かったこと |
|---|---|
| 表を作る（1節） | 上位10ジャンル×9年代の表。合計18,544本だが、映画は9,162本 |
| 独立なら何本？（2節） | ？（このあとの段で） |
| ずれを測る（3節） | ？（このあとの段で） |
| 地図に縮める（4節） | ？（このあとの段で） |
| 倍率で確かめる（5節） | ？（このあとの段で） |

まだ分からないのは、偏りがなければ各マスが何本になるはずか、です。


---


## 2. 偏りがなければ何本？


ここから、段2「独立なら何本？」（2節）です。問いは、年代とジャンルに関係がなければ、各マスは何本になるはずか、です。

題名から年代を取り出し、ジャンルを分けて、10ジャンル×9年代の表を作りました。表の合計は18,544で、重ねて数えています。年代の本数も、ジャンルの本数もばらばらなので、そのままでは比べられません。2000年代のドラマが多いのは、2000年代の映画が多いからかもしれず、ドラマ自体が多いジャンルだからかもしれません。


### 期待度数

年代$i$の合計を$n_{i\cdot}$、ジャンル$j$の合計を$n_{\cdot j}$、全体を$n$とします。

$$
E_{ij} = \frac{n_{i\cdot} \times n_{\cdot j}}{n}
$$

年代とジャンルが関係なければ（独立なら）、マスの数はおよそ$E_{ij}$になります。年代の割合 × ジャンルの割合 × 全体です。


### お手本：期待度数

- 役割：年代の合計とジャンルの合計から、期待度数の表を作る
- 入力：観測度数の表`O`
- 出力：期待度数の表`E`（9×10）


In [ ]:
n = O.values.sum()
E = np.outer(O.sum(axis=1), O.sum(axis=0)) / n
E = pd.DataFrame(E, index=O.index, columns=O.columns)
print(E.shape)
print(E.iloc[0, :3].round(1).tolist(), O.iloc[0, :3].tolist())


#### 1行ずつ読む

- 1行目：表の合計
- 2行目：年代の合計（9個）と、ジャンルの合計（10個）のすべての組を掛けて（`np.outer`）、合計で割る
- 3行目：行と列の名前を付けて、表にする
- 4・5行目：9×10の期待度数。1930年代の先頭3ジャンルの、期待と観測


#### 中身を見る：np.outer


In [ ]:
print(np.outer([1, 2, 3], [10, 20]))


左の3個と右の2個の、すべての組の積を並べた3×2の表です。入れ子の`for`（第9回）を1行で書いたのと同じです。


### お手本：倍率の表

観測度数 ÷ 期待度数を、倍率と呼ぶことにします。1より大きければ多すぎ、小さければ少なすぎです。年代やジャンルの大きさの違いを、取り除いた比べ方です。


In [ ]:
ratio = O / E
print(ratio.loc[1930, 'Romance'].round(2))
print(O.loc[1930, 'Romance'], E.loc[1930, 'Romance'].round(1))


#### 1行ずつ読む

- 1行目：マスごとに、観測÷期待（表どうしの割り算は、同じ位置どうし）
- 2行目：1930年代のロマンスの倍率（`loc`で行と列の名前を指す）
- 3行目：その観測度数と期待度数


In [ ]:
ratio.round(2)


古い年代はロマンス・ドラマが多く、1940年代のSFはほとんどありません。


> 重要：本数そのままでは、年代やジャンルの大きさに引きずられる。偏りがなければ何本か（期待度数）と比べると、割合の偏りが読める。


### TODO②：見るマスを替える（予想 → 実行 → 比べる）

下のセルは、1つのマスの倍率と観測度数を表示します。

まず、予想を書いてください。1940年代のSFは、期待と比べて何倍でしょうか。


（ここに予想を書く）


予想を書いたら、セルの`1930`を`1940`に、`'Romance'`を`'Sci-Fi'`に替えて実行してください（2か所とも）。


In [ ]:
# TODO: 1930 を 1940 に、'Romance' を 'Sci-Fi' に替えて実行する

print(ratio.loc[1940, 'Sci-Fi'].round(2))
print(O.loc[1940, 'Sci-Fi'])


### やってみよう：予想と比べる

実行した結果を、予想と比べてください。何倍でしたか。観測は何本でしたか。


（ここに答えを書く）


1940年代のSFは0.11倍です。下のセルで期待度数を見ると17.5本で、それに対して2本しかありません。ただし、2本と少ないので、倍率だけでは偶然の揺れか見分けにくいのです。


In [ ]:
print(E.loc[1940, 'Sci-Fi'].round(1))


### ここまでで分かったこと

| 段 | 分かったこと |
|---|---|
| 表を作る（1節） | 上位10ジャンル×9年代の表。合計18,544本だが、映画は9,162本 |
| 独立なら何本？（2節） | 期待度数と比べた倍率：1930年代のロマンス2.32倍 |
| ずれを測る（3節） | ？（このあとの段で） |
| 地図に縮める（4節） | ？（このあとの段で） |
| 倍率で確かめる（5節） | ？（このあとの段で） |

まだ分からないのは、どのマスのずれが大きいと言えるか、です。


---


## 3. どのマスがずれている？


ここから、段3「ずれを測る」（3節）です。問いは、どのマスのずれが大きいか。本数の少ないマスと、多いマスを同じ物差しで比べるには、です。

偏りがなければ何本か（期待度数）を出し、倍率で比べました。でも、期待が2本のマスの3倍と、期待が500本のマスの1.2倍は、どちらが大きなずれでしょうか。


### 標準化残差

$$
r_{ij} = \frac{O_{ij} - E_{ij}}{\sqrt{E_{ij}}}
$$

観測と期待の差を、期待の平方根で割ります。数え上げの揺れの大きさで割った、ずれの大きさです。偏りがなければ、多くのマスで−2〜2くらいに収まります。


### お手本：標準化残差

- 役割：マスごとの標準化残差と、その2乗の和を出す
- 入力：`O`と`E`
- 出力：標準化残差の表`R`


In [ ]:
R = (O - E) / np.sqrt(E)
print(R.loc[1930, 'Romance'].round(1), R.loc[1940, 'Sci-Fi'].round(1))
print((R.values ** 2).sum().round(1))


#### 1行ずつ読む

- 1行目：マスごとに、（観測−期待）÷√期待
- 2行目：1930年代のロマンスと、1940年代のSF
- 3行目：すべてのマスの2乗の和（カイ二乗統計量）


In [ ]:
plt.figure(figsize=(10, 5))
plt.imshow(R, cmap='bwr', vmin=-6, vmax=6)
plt.xticks(range(10), O.columns, rotation=45)
plt.yticks(range(9), O.index)
plt.colorbar()
plt.show()


赤は多すぎ、青は少なすぎです。赤い左上（古い年代のロマンス・ドラマ）と、右下の赤（2010年代のアクション）が目立ちます。


### お手本：大きいずれを並べる


In [ ]:
Rs = R.stack().sort_values()
print(Rs.tail(3).round(1))


#### 1行ずつ読む

- 1行目：表を1列に並べ（`stack`）、小さい順に並べ替える
- 2行目：大きい3つ


### カイ二乗統計量

標準化残差の2乗の和548.4は、表全体のずれの大きさです。教科書では、この値でカイ二乗検定をして、偶然かどうかを確かめます。ところが、この表には、その検定の前提が当てはまりません。


> 重要：カイ二乗検定の式は、表の1つの数が1本の別の映画、という前提に立つ。重ねて数えた表では前提が崩れるので、p値（p-value）は使わず、残差でずれを読む。


> 今日の重要概念（統計）：期待と比べて、残差で読む
>
> 区分×区分の表は、偏りがなければ何本か（期待度数）と比べ、ずれを標準化残差で読む。本数の大小に引きずられずに、どこが多すぎ・少なすぎかが分かる。


### TODO③：少なすぎのほうを見る（予想 → 実行 → 比べる）

下のセルは、標準化残差を小さい順に並べた`Rs`の、最後の3つ（大きいほう）を表示します。

まず、予想を書いてください。いちばん少なすぎるマスはどこでしょうか。


（ここに予想を書く）


予想を書いたら、セルの`tail`を`head`に替えて実行してください。


In [ ]:
# TODO: tail を head に替えて実行する

print(Rs.head(3).round(1))


### やってみよう：予想と比べる

実行した結果を、予想と比べてください。どのマスでしたか。


（ここに答えを書く）


いちばん少なすぎるのは、2010年代のロマンス（−5.7）です。1950年代のアクション、2010年代のドラマが続きます。


### ここまでで分かったこと

| 段 | 分かったこと |
|---|---|
| 表を作る（1節） | 上位10ジャンル×9年代の表。合計18,544本だが、映画は9,162本 |
| 独立なら何本？（2節） | 期待度数と比べた倍率：1930年代のロマンス2.32倍 |
| ずれを測る（3節） | 標準化残差の2乗の和548.4。いちばん大きいのは1930年代のロマンス |
| 地図に縮める（4節） | ？（このあとの段で） |
| 倍率で確かめる（5節） | ？（このあとの段で） |

まだ分からないのは、9×10のずれの表全体を、1枚にまとめて見られるか、です。


---


## 4. 1枚の地図にできるか？


ここから、段4「地図に縮める」（4節）です。問いは、9×10のずれの表を、1枚の地図にまとめられるか、です。

標準化残差で、どのマスが多すぎ・少なすぎかを読みました。表は90マスあり、全体の形をつかむには多すぎます。第11回の主成分分析（principal component analysis）と同じく、特異値分解で縮めます。

ずれ（標準化残差）の表を、特異値分解で少ない次元に縮めると、行（年代）と列（ジャンル）を、同じ地図に置けます。これを対応分析（correspondence analysis）といいます。近くにある年代とジャンルは、その組が多すぎる（正のずれ）組です。


### お手本：残差の表を特異値分解

- 役割：標準化残差の表を特異値分解し、次元ごとの寄与率（proportion of variance explained）を出す
- 入力：`R`
- 出力：`U`・`s`・`Vt`


In [ ]:
U, s, Vt = np.linalg.svd(R.values, full_matrices=False)
print(U.shape, s.shape, Vt.shape)
print((s ** 2).sum().round(1))
print((s ** 2 / (s ** 2).sum())[:3].round(3))


#### 1行ずつ読む

- 1行目：9×10の残差の表を特異値分解（第11回）。`U`が年代の向き、`Vt`がジャンルの向き、`s`が特異値（singular value）
- 2行目：形
- 3行目：特異値の2乗の和は548.4。カイ二乗統計量（chi-squared statistic）と同じ
- 4行目：次元ごとの寄与率。第1の次元だけで0.673

2つの次元で、ずれの84%を持ちます。


### お手本：地図の座標

- 役割：年代とジャンルの、地図の上の座標を出す
- 入力：`U`・`s`・`Vt`と、年代・ジャンルの合計
- 出力：`row_xy`（年代）と`col_xy`（ジャンル）


In [ ]:
row_n = O.sum(axis=1).values.reshape(9, 1)
col_n = O.sum(axis=0).values.reshape(10, 1)
row_xy = U * s / np.sqrt(row_n)
col_xy = Vt.T * s / np.sqrt(col_n)
print(row_xy[:3, :2].round(2))


#### 1行ずつ読む

- 1・2行目：年代の合計とジャンルの合計を、縦1列の形にする（`reshape`、第11回）
- 3・4行目：向き×特異値を、合計の平方根で割る。本数の多い行や列が、原点から離れすぎないようにする
- 5行目：先頭3年代の、第1・第2の次元の座標


### お手本：地図を描く


In [ ]:
dy = 1
plt.figure(figsize=(10, 7))
plt.scatter(row_xy[:, 0], row_xy[:, dy])
for i in range(9):
    plt.text(row_xy[i, 0], row_xy[i, dy], O.index[i])
plt.scatter(col_xy[:, 0], col_xy[:, dy], marker='s')
for j in range(10):
    plt.text(col_xy[j, 0], col_xy[j, dy], O.columns[j])
plt.axhline(0, color='gray')
plt.axvline(0, color='gray')
plt.show()


#### 1行ずつ読む

- 1・2行目：図を大きくし、縦軸に使う次元の番号を決める（1は第2の次元）
- 3〜5行目：年代を点で打ち、名前を書く
- 6〜8行目：ジャンルを四角で打ち、名前を書く
- 9〜11行目：原点を通る線を引いて表示する

右に1930〜50年代とロマンス・ドラマ、左に2010年代とSF・アクション・ホラーが並びます。横軸は、古い↔新しいです。

えっ、本数がいちばん多い2000年代（5,809）は、原点の近くにいます。原点は「全体の平均の割合」です。2000年代は本数が多いので、全体の割合にいちばん近いのです。地図は、本数ではなく、割合の偏りを表します。


> 重要：対応分析の地図で原点から遠いほど、全体の割合からずれている。年代とジャンルが同じ向きにあれば、その組が多すぎる。本数の多さは、地図には出ない。


### TODO④：縦軸を第3の次元に（予想 → 実行 → 比べる）

下のセルは、横に第1の次元、縦に第`dy + 1`の次元をとって地図を描きます。

まず、予想を書いてください。第3の次元（寄与率0.082）では、どの年代が離れるでしょうか。


（ここに予想を書く）


予想を書いたら、セルの`1`を`2`に替えて実行してください（`dy = 1`の行）。


In [ ]:
# TODO: 1 を 2 に替えて実行する

dy = 2
plt.figure(figsize=(10, 7))
plt.scatter(row_xy[:, 0], row_xy[:, dy])
for i in range(9):
    plt.text(row_xy[i, 0], row_xy[i, dy], O.index[i])
plt.scatter(col_xy[:, 0], col_xy[:, dy], marker='s')
for j in range(10):
    plt.text(col_xy[j, 0], col_xy[j, dy], O.columns[j])
plt.axhline(0, color='gray')
plt.axvline(0, color='gray')
plt.show()


### やってみよう：予想と比べる

実行した結果を、予想と比べてください。どの年代が離れましたか。


（ここに答えを書く）


第3の次元では、1930年代が上に、1970年代が下に離れます。寄与率は0.082と小さく、細かい違いです。


### ここまでで分かったこと

| 段 | 分かったこと |
|---|---|
| 表を作る（1節） | 上位10ジャンル×9年代の表。合計18,544本だが、映画は9,162本 |
| 独立なら何本？（2節） | 期待度数と比べた倍率：1930年代のロマンス2.32倍 |
| ずれを測る（3節） | 標準化残差の2乗の和548.4。いちばん大きいのは1930年代のロマンス |
| 地図に縮める（4節） | 横軸（0.673）は古い↔新しい。2000年代は原点の近く |
| 倍率で確かめる（5節） | ？（このあとの段で） |

まだ分からないのは、地図で見えたことが、表の数でも本当か、です。


---


## 5. 地図は本当か？


ここから、段5「倍率で確かめる」（5節）です。問いは、地図で近く見えた年代とジャンルは、表の数でも本当に多すぎるか、です。

残差の表を特異値分解で縮め、年代とジャンルを1枚の地図に置きました。横軸は古い↔新しいで、本数の多い2000年代は真ん中でした。地図は縮めた近似なので、最後は、表の数（倍率）で確かめます。


### お手本：1つの年代の倍率


In [ ]:
print(ratio.loc[1930].sort_values(ascending=False).round(2))


1930年代の倍率を、大きい順に並べました。1930年代は、地図でロマンス・ドラマの近くにいました。倍率でも、ロマンス2.32倍、ドラマ1.31倍です。地図で反対側にいたSF（0.24倍）・アクション（0.34倍）は、下のほうにいます。


### join：最小の例

9つの年代すべてで、倍率の上位3ジャンルを並べたいところです。年代ごとに同じ手順をくり返すので`for`で回し、ジャンルの名前のリストを「、」でつないで1つの文字列にします（`join`）。


In [ ]:
names = ['Romance', 'Drama', 'Comedy']
print('、'.join(names))
print('1930年代：' + '、'.join(names))


`'、'.join(リスト)`は、リストの文字列を「、」でつなぎます。`split`の逆です。`+`で文字列どうしをつなぎます。


### お手本：年代ごとの1文

- 役割：年代ごとに、倍率の上位3ジャンルを1文にして表示する
- 入力：倍率の表`ratio`
- 出力：9行の文


In [ ]:
for d in ratio.index:
    top3 = ratio.loc[d].sort_values(ascending=False).index[:3]
    print(str(d) + '年代：' + '、'.join(top3))


#### 1行ずつ読む

- 1行目：年代を1つずつ取る
- 2行目：その年代の倍率を大きい順に並べ、上位3つのジャンルの名前を取る
- 3行目：年代を文字にし（`str`）、ジャンルを「、」でつないで、1文にする

1930〜50年代はロマンス・ドラマ、1970〜80年代はホラー・SF、2010年代はアクション・SF・冒険（Adventure）です。地図の横軸の、右から左への流れと同じです。


> 重要：対応分析の地図は、ずれの表を縮めた近似。地図で近く見えた組は、倍率や残差の表に戻って、数で確かめてから言う。


### TODO⑤：上位の数を替える（予想 → 実行 → 比べる）

下のセルは、年代ごとに倍率の上位のジャンルを1文にします。

まず、予想を書いてください。上位5つにすると、1990・2000年代の1文はどうなるでしょうか。


（ここに予想を書く）


予想を書いたら、セルの`3`を`5`に替えて実行してください。


In [ ]:
# TODO: 3 を 5 に替えて実行する

for d in ratio.index:
    top = ratio.loc[d].sort_values(ascending=False).index[:5]
    print(str(d) + '年代：' + '、'.join(top))


### やってみよう：予想と比べる

実行した結果を、予想と比べてください。1990・2000年代は、どんなジャンルが並びましたか。


（ここに答えを書く）


下のセルで、1990・2000年代の上位5つの倍率を確かめられます。上位5つでも倍率は1.0〜1.2くらいで、どのジャンルもほぼ全体の割合どおりです。地図で原点の近くにいたのと合います。


In [ ]:
for d in [1990, 2000]:
    print(d, ratio.loc[d].sort_values(ascending=False)[:5].round(2).tolist())


### ここまでで分かったこと

| 段 | 分かったこと |
|---|---|
| 表を作る（1節） | 上位10ジャンル×9年代の表。合計18,544本だが、映画は9,162本 |
| 独立なら何本？（2節） | 期待度数と比べた倍率：1930年代のロマンス2.32倍 |
| ずれを測る（3節） | 標準化残差の2乗の和548.4。いちばん大きいのは1930年代のロマンス |
| 地図に縮める（4節） | 横軸（0.673）は古い↔新しい。2000年代は原点の近く |
| 倍率で確かめる（5節） | 古い年代はロマンス・ドラマ、1970〜80年代はホラー・SF、2010年代はアクション |

5つの段がそろい、今日の問いに答えられます。


### 答え

古い年代はロマンス・ドラマ、1970〜80年代はホラー・SF、2010年代はアクション・SFが、割合として多くなっています。地図の横軸（寄与率0.673）は、この時代の流れで、本数の多い1990・2000年代は真ん中でした。区分×区分の表は、期待度数と比べた残差と倍率で読みました。


まだ言えないこと

- ずれが偶然でないか。重ねて数えた表なので、カイ二乗検定の式は使えない
- MovieLensの一覧は、評価サイトの利用者が見た映画。作られた映画のすべてではない
- ジャンルの付け方は、サイトの決め方による


### レポートなら、こう書く

今日の解析を、期末レポートの項目に写すと、次のようになります。

- データ：MovieLensの映画の一覧から、1930〜2010年代の9,682本。ジャンルを分けて、上位10ジャンル×9年代の表にした
- 注意：表の合計は18,544で、映画は9,162本。重ねて数えているので検定の式は使わず、残差で読んだ
- ずれ：1930年代のロマンスは期待の2.32倍（残差6.0）、2010年代のロマンスは残差−5.7
- 地図：残差の表を特異値分解。第1の次元（寄与率0.673）は古い↔新しい
- 驚き：本数がいちばん多い2000年代は原点の近く。地図は本数でなく割合の偏りを表す
- 限界：評価サイトの利用者が見た映画だけ。ジャンルの付け方もサイトによる


---


## 目標に答えられたか


- 今日の目標は「映画のジャンルの好まれ方は、年代でどう違うか。それを1枚の地図にできるか」でした。倍率の表と地図で答えられますか？
- 1節で、`split`と`explode`は、それぞれ何をしましたか？　表の合計が映画の本数より多いのはなぜですか？
- 2節で、本数そのままでなく期待度数と比べたのはなぜですか？
- 3節で、カイ二乗検定のp値を使わなかったのはなぜですか？
- 4節で、本数がいちばん多い2000年代が原点の近くにいたのはなぜですか？
- 5節で、地図で見えたことを、何で確かめましたか？


---


## 今日の要点


- 今日、持ち帰る1文：区分×区分の表は、偏りがなければ何本か（期待度数）と比べ、標準化残差でずれを読む。ずれの表を特異値分解すると、行と列を1枚の地図に置ける。地図で見えたことは、倍率の表で確かめる
- 重要概念（統計）：期待と比べて、残差で読む。区分×区分の表は、期待度数と比べ、ずれを標準化残差で読む
- 重要概念（Python）：文字列を分ける。1つのマスに詰めた複数の値は、区切りの文字で分け（split）、1つずつを1行にする（explode）
- 表の合計が、数えたもの（映画）の数と合うかを確かめる。重ねて数えていたら、検定の式は使えない
- 対応分析の地図の近さは、本数ではなく割合の偏り


---


## 次回へ


今日の表は、年代とジャンルの組を数えたものでした。では、もの同士のつながり（誰と誰が、どこからどこへ）そのものは、どう分析すればよいでしょうか。


---


## 課題（提出）


提出するもの：絵2枚（画像ファイル）。今日の重要概念の、統計とPythonを1枚ずつです。提出フォームから出してください。期限はありません。

コードは書きません。上手な絵である必要はありません。紙に手で描いて写真を撮っても、描画アプリで描いてもかまいません。


### 今日の重要概念（2つ）

- 統計：期待と比べて、残差で読む。区分×区分の表は、偏りがなければ何本か（期待度数）と比べ、ずれを標準化残差で読む（2・3節）
- Python：文字列を分ける。1つのマスに詰めた複数の値は、区切りの文字で分け（split）、1つずつを1行にする（explode）（1節）


### 今日の2枚（絵にする）

2つの重要概念を、1枚ずつ絵にしてください。その考えが1枚で伝わる絵です。数式やコードは描かなくてかまいません。半年後の自分が見て、何の話だったか思い出せる絵を目指します。


<details><summary>詰まったら</summary>

期待と比べて残差で読む、なら、たとえば「クラスごとの給食の投票」を描けます。
大きいクラスは、どの給食も票が多くなります。票の数だけでは、クラスの好みは分かりません。「クラスの人数 × 全校での人気」で、偏りがなければ何票かを出し、それより多い・少ないで読みます。2節の倍率の表が手本になります。

</details>


<details><summary>詰まったら</summary>

文字列を分けるなら、たとえば「買い物メモを切り分ける」を描けます。
1枚の紙に「りんご・みかん・ぶどう」と書いた買い物メモを、「・」で切り分けて、1つずつの紙にします。3枚になった紙を、果物ごとに数えます。1節のトイ・ストーリーが5行になる例が手本になります。

</details>


---


## 発展（任意）


### 区分が3つ以上なら？（多重対応分析）

今日は、年代とジャンルの2つの区分の表でした。区分が3つ以上（たとえば第9回のペンギンの種・島・性別）でも、同じ考えで地図にできます。1羽を、区分の値ごとの0と1の列にした表を、同じ特異値分解で縮めます。


In [ ]:
# データはこのリポジトリから読み込む
DATA_DIR = ('https://raw.githubusercontent.com/k0heiun0/'
            'applied_exercise_preview/main/shared/data')
pen = pd.read_csv(f'{DATA_DIR}/penguins.csv').dropna(subset=['sex'])
D = pd.get_dummies(pen[['species', 'island', 'sex']],
                   prefix='', prefix_sep='').astype(int)
print(D.shape)
D.head(3)


#### 1行ずつ読む

- `DATA_DIR`と`pd.read_csv`の行：第9回のペンギンを読み、性別が欠測の羽を外す
- `get_dummies`の行：種・島・性別の値ごとに、当てはまれば1、そうでなければ0の列を作る
- `print`と`head`の行：333羽×8列（種3・島3・性別2）


In [ ]:
P = D.values / D.values.sum()
r = P.sum(axis=1)
c = P.sum(axis=0)
S = (P - np.outer(r, c)) / np.sqrt(np.outer(r, c))
U2, s2, Vt2 = np.linalg.svd(S, full_matrices=False)
xy = Vt2.T * s2 / np.sqrt(c).reshape(8, 1)
plt.scatter(xy[:, 0], xy[:, 1])
for j in range(8):
    plt.text(xy[j, 0], xy[j, 1], D.columns[j])
plt.axhline(0, color='gray')
plt.axvline(0, color='gray')
plt.show()


#### 1行ずつ読む

- 1〜3行目：表を全体の和で割って割合にし、行の和と列の和を出す
- 4行目：期待からのずれを、期待の平方根で割る（標準化残差と同じ考え）
- 5・6行目：特異値分解して、列（区分の値）の座標を出す
- 7〜12行目：地図に描く

Gentoo（ジェンツー）とBiscoe（ビスコー島）、Chinstrap（ヒゲ）とDream（ドリーム島）が近くにいます。種がどの島にいるかが、地図に出ました。性別（male・female）は真ん中です。

試すなら、`'sex'`を外して、種と島の2つだけで地図にしてみましょう。
